# planB — train all arms on GPU, 30 epochs, then run the counterfactualTrains seven arms, evaluates them on the real test set, runs the planA-protocolcounterfactual, and downloads everything back to your machine as one zip.| arm | backbone | head | gap | notes ||---|---|---|---|---|| `unconstrained` | BiLSTM | none | 3 h | reference: what no constraint layer looks like || `rayen` | BiLSTM | anchor + ray | 3 h | || `hardnet` | BiLSTM | exact projection | 3 h | || `brits` | BRITS | exact projection | 3 h | || `brits_rayen` | BRITS | anchor + ray | 3 h | **new** — completes the 2x2 || `blackout` | BiLSTM | exact projection | **whole day** | **new** — 288-step gap || `saits` | SAITS | exact projection | 3 h | **new** — see the note below |All arms share: hidden 192, **one MSE** over 8 z-scored channels, ReLU in z-spacefor curtailment, split output heads, no curtailment credit, **p99.9 rampenvelope**, and — restored in this run — the **interpolation-residualparameterisation**.### Two things changed since the last local run, both of which should move the numbers**1. The nd feature is now supply-side.** The models were reading`net_demand = demand - wind - solar - curtailment` while the head balanced them to`SIGN.dispatch`. Those differ by up to 4,974 MW (+1,104 MW on 2026-06-04). Nowthey are the same quantity.**2. The interpolation residual is back.** planA parameterised the fill as`interp + deviation`; planB had been predicting the level directly, forcing everyarm to learn a whole trajectory from scratch instead of a correction to thestrongest baseline available. Smoke test: `hardnet` val 3.45 -> 1.57 from thischange alone. It does not apply to the `rayen` arms, whose 6 outputs are adirection rather than a level — their anchor already plays that role.

## Why SAITS is the seventh armThe problem planB measured: every arm trails linear interpolation (50.6 MW), andthe counterfactual allocation is decided by the constraint head rather thanlearned — `rayen`'s anchor alone scores 63.7 MW against 82.8 with the network,so the network is making it *worse*.SAITS (Du, Cote & Liu 2023, *Expert Systems with Applications* 219:119619) hasthree properties aimed at exactly that:1. **Diagonally-masked self-attention.** A timestep cannot attend to itself, so   its estimate must be assembled from elsewhere. This is BRITS's zeroed-diagonal   feature regression generalised to the time axis. A mid-gap step reaches *both*   pinned boundaries in one hop; BRITS propagates 18 recurrent steps to do the   same, which is why it needs a decay term at all.2. **Two cascaded blocks with a learned combination.** The second re-imputes given   the first's completion, and the two are mixed by a weight from the attention   map and the mask. A single pass has to commit; this does not.3. **Attention to the drivers at the same timestep.** Net demand, wind and solar   at step *t* constrain the dispatch at step *t* directly, without a recurrent   path.**Deviation to record:** SAITS also trains with MIT (masked imputation task —randomly mask observed cells and reconstruct them). That is not implemented here,because the constraint head only operates on the contiguous gap slice, so MITcells in the context cannot pass through it without a second loss term — and thebrief was one MSE. MIT is the obvious next thing to try if SAITS underperforms.

In [ ]:
# ---- 1. clone + GPU check -------------------------------------------------REPO  = "github.com/nm-quan/energy_modelling.git"TOKEN = ""   # <-- private repo: paste a fine-grained READ-ONLY token (Contents: read)import os, subprocess, sysos.chdir("/content")url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"if not os.path.exists("/content/energy_modelling"):    subprocess.run(["git", "clone", "-q", url], check=True)os.chdir("/content/energy_modelling")subprocess.run(["git", "pull", "-q"])import torchname = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"mem  = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0print(f"torch {torch.__version__} | {name} | {mem:.0f} GB")# Batch size is the lever that matters here. Both constraint heads run a Python# loop over the 36 gap steps ONCE PER BATCH, so its cost is amortised linearly:# doubling the batch nearly halves per-window head time. Memory is not the# constraint -- the whole 40k window set is ~450 MB and lives on the GPU.if "A100" in name or mem > 60:    BATCH, FLAGS = 1024, "--compile"     # A100 80GBelif "A100" in name or mem > 35:    BATCH, FLAGS = 768, "--compile"      # A100 40GBelif mem > 20:    BATCH, FLAGS = 512, ""               # L4 / V100 32GBelif mem > 0:    BATCH, FLAGS = 256, ""               # T4 16GBelse:    BATCH, FLAGS = 128, ""    print("\n!! Runtime > Change runtime type > GPU, then re-run this cell.")OPTS = f"--batch {BATCH} {FLAGS}".strip()print(f"\nbatch={BATCH}  flags={FLAGS or 'none'}")print("  --compile  : backbone only; heads stay eager (data-dependent control flow)")print("  amp is OFF : bf16 in the head would put balance error in the tens of MW")import numpy as npz = np.load("data/preprocessed/hist/5min/net_dispatch_ren/prepared.npz")print("\nflats:", z["Xtr_flat"].shape, "| features:", len(z["feat_cols"]))

In [ ]:
# ---- 2. the constraint envelope + head self-tests -------------------------# p99.9 ramps, empirical-max caps. Confirms 0/400 windows lose bridge feasibility# and that both heads are exact before a single epoch is spent.!python3 planB/limits.pyprint("\n" + "=" * 70 + "\n")!python3 planB/heads.py

## TrainingOne cell per arm so partial progress survives a disconnect. Early stopping is`--patience 10` on validation loss, so most arms will stop before 30 epochs.`blackout` masks 288 steps instead of 36, so its constraint head loops 8x as manytimes per window. It uses fewer training windows to compensate — change`--n-train` if you have the session time.

### A100 / GPU notes`OPTS` is set automatically by the cell above. What it does and why:- **Batch size is the main lever.** Both heads walk the 36 gap steps in a Python  loop *once per batch*, so that cost amortises linearly — batch 1024 does the  same 36 iterations as batch 128 for 8x the windows. Memory is not the binding  constraint: the whole 40k window set is ~450 MB and is moved onto the GPU once,  which also removes a host-to-device copy per batch.- **`--compile` wraps the BACKBONE only.** The heads carry data-dependent control  flow — a 36-step sweep with a 40-step bisection inside — which either  graph-breaks or unrolls into an enormous graph. Expect a slow first batch.  Helps SAITS and BRITS most; `nn.LSTM` least, since cuDNN already fuses it.- **TF32 is on.** Safe here: the heads are elementwise (clamp / add / mul / sum),  not matmul, so the exact arithmetic is untouched.- **AMP is deliberately off.** `--amp` exists and confines bf16 to the backbone,  but even so it is not the default: the whole point of the constraint layer is  balance exact to ~1e-12, and bf16 anywhere near it puts the residual in the  tens of MW on a 5 GW stack. Turn it on only if you are trading that away  knowingly.If you want to raise the batch further, raise `--lr` with it — these runs useAdamW at 1e-3 tuned for batch 128.

In [ ]:
# ---- unconstrained — reference: no constraint layer!python3 planB/fit.py --arm unconstrained --epochs 30 --patience 10 $OPTS

In [ ]:
# ---- rayen — anchor + ray!python3 planB/fit.py --arm rayen --epochs 30 --patience 10 $OPTS

In [ ]:
# ---- hardnet — exact projection!python3 planB/fit.py --arm hardnet --epochs 30 --patience 10 $OPTS

In [ ]:
# ---- brits — BRITS + projection!python3 planB/fit.py --arm brits --epochs 30 --patience 10 $OPTS

In [ ]:
# ---- brits_rayen — BRITS + ray  (new)!python3 planB/fit.py --arm brits_rayen --epochs 30 --patience 10 $OPTS

In [ ]:
# ---- saits — SAITS + projection  (new)!python3 planB/fit.py --arm saits --epochs 30 --patience 10 $OPTS

In [ ]:
# ---- blackout — whole-day gap  (new)!python3 planB/fit.py --arm blackout --epochs 30 --patience 10 $OPTS --n-train 15000

## Evaluation and counterfactual

In [ ]:
# ---- per-channel MAE / MRE on 400 random test 3h gaps, + feasibility -------!python3 planB/eval.py --n 400

In [ ]:
# ---- planA-protocol counterfactual, 10 days, all arms ---------------------# free window 11:00-14:00, rebound 2.4% / reduce 2.4%, scaled + masked modes,# project_full_day over all 288 steps. Writes counterfactual.md + a stacked figure.!python3 planB/counterfactual.py --days 10

In [ ]:
# ---- loss curves, all arms ------------------------------------------------import json, matplotlib.pyplot as pltfrom pathlib import PathOUT = Path("planB/results")fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))for p in sorted(OUT.glob("*_history.json")):    if "smoke" in p.name:        continue    h = json.loads(p.read_text()); ep = range(1, len(h["train"]) + 1)    ax[0].plot(ep, h["train"], lw=2, label=h["arm"])    ax[1].plot(ep, h["val"], lw=2, label=h["arm"])for a, t in zip(ax, ("train", "validation")):    a.set_title(f"{t} loss (one MSE, 8 z-scored channels)", loc="left")    a.set_xlabel("epoch"); a.grid(alpha=0.2); a.legend(fontsize=8, frameon=False)fig.tight_layout(); fig.savefig(OUT / "loss_curves.png", dpi=140, facecolor="white")plt.show()

In [ ]:
# ---- show the counterfactual figure inline --------------------------------from IPython.display import Image, display, Markdownfrom pathlib import PathOUT = Path("planB/results")for f in ("counterfactual_10day.png", "loss_curves.png"):    if (OUT / f).exists():        display(Image(str(OUT / f)))for f in ("eval.md", "counterfactual.md"):    if (OUT / f).exists():        display(Markdown((OUT / f).read_text()))

## Download everything back to your machineUnzip into `planB/results/` locally. Contains the weights, per-epoch history, theeval and counterfactual reports, and the figures.

In [ ]:
# ---- zip + download -------------------------------------------------------import shutilfrom google.colab import filesshutil.make_archive("/content/planB_results", "zip", "planB/results")sz = os.path.getsize("/content/planB_results.zip") / 1e6print(f"planB_results.zip  {sz:.1f} MB")files.download("/content/planB_results.zip")

### If the session drops mid-runWeights and history are written after each arm finishes, so re-running only theunfinished training cells is enough. The eval and counterfactual cells skip anyarm whose `.pt` is missing rather than failing.### What to look for in the results- **Does any arm beat interpolation** (50.6 MW dispatch MAE, 43.0 curtailment)?  Nothing has yet, and one epoch was the standing excuse. Thirty removes it.- **Does `rayen` beat its own anchor?** Re-run the anchor ablation locally. If it  never does, the anchor is the model and the ray head is not earning its place.- **`hardnet` vs `brits` vs `saits`** share the identical head and envelope, so  differences between them are the backbone alone.- **`brits` vs `brits_rayen`** and **`hardnet` vs `rayen`** are the same head  swap on two backbones — if the head effect has the same sign on both, it is the  head; if not, it is an interaction.- **Coal share in the counterfactual** against the real grid's 43.4%. Every arm  missed badly in both directions at one epoch.